# Hands-on Lab Evaluation - 4 (Module IV)
### Topic: Clustering & Association Rule Mining
**Student Implementation:** Mall Customer Segmentation & Groceries Market Basket Analysis

> **Note:** Datasets are loaded directly from the GitHub repository raw links so this notebook runs seamlessly anywhere (locally, Jupyter, or Google Colab).

In [ ]:
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

## Task 1: Load and Explore Mall Customer Dataset from GitHub

In [ ]:
# Load Mall Customer dataset directly through GitHub URL
mall_url = 'https://raw.githubusercontent.com/ayankhan96686-create/ml-hand-on-lab-4/main/Mall_Customers.csv'
try:
    mall = pd.read_csv(mall_url)
    print('Loaded Mall Customers dataset from GitHub successfully!')
except Exception:
    mall = pd.read_csv('Mall_Customers.csv')
    print('Loaded from local file fallback.')

# First 10 and last 10 rows
print('\nFirst 10 rows:')
display(mall.head(10))

print('\nLast 10 rows:')
display(mall.tail(10))

# Dataset info
print('\nDataset Info:')
mall.info()

# Summary statistics
print('\nSummary Statistics:')
display(mall.describe())

# Check missing values
print('\nMissing values count:')
print(mall.isnull().sum())

In [ ]:
# Visualize feature distributions
plt.figure(figsize=(10, 8))

plt.subplot(2, 2, 1)
sns.histplot(mall['Age'], kde=True, color='skyblue')
plt.title('Age Distribution')

plt.subplot(2, 2, 2)
sns.histplot(mall['Annual Income (k$)'], kde=True, color='salmon')
plt.title('Annual Income (k$)')

plt.subplot(2, 2, 3)
sns.histplot(mall['Spending Score (1-100)'], kde=True, color='lightgreen')
plt.title('Spending Score (1-100)')

plt.subplot(2, 2, 4)
sns.countplot(data=mall, x='Gender', palette='Set2', hue='Gender', legend=False)
plt.title('Gender Count')

plt.tight_layout()
plt.show()

## Task 2: Data Preprocessing and Feature Scaling

In [ ]:
# Handle missing values if any
if mall.isnull().sum().sum() > 0:
    mall.dropna(inplace=True)

# Select Annual Income and Spending Score for clustering
X = mall[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Scale features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Features scaled successfully. Shape:', X_scaled.shape)

## Task 3: K-Means Clustering and Elbow Method

In [ ]:
# Calculate WCSS for k=1 to 10
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

# Plot elbow curve
plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), wcss, marker='o', color='blue')
plt.title('Elbow Method for K-Means')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS / Inertia')
plt.axvline(x=5, color='red', linestyle='--', label='Elbow at k=5')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with optimal k=5
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
mall['KMeans_Cluster'] = kmeans_labels

km_score = silhouette_score(X_scaled, kmeans_labels)
print(f'K-Means (k=5) Silhouette Score: {km_score:.4f}')

# Plot clusters
plt.figure(figsize=(8, 5))
colors = ['red', 'blue', 'green', 'purple', 'orange']
labels = [
    'Careless (Low Income, High Spend)',
    'Careful (High Income, Low Spend)',
    'Sensible (Low Income, Low Spend)',
    'Affluent (High Income, High Spend)',
    'Standard (Average Income & Spend)'
]

for i in range(5):
    plt.scatter(X[kmeans_labels == i, 0], X[kmeans_labels == i, 1], s=50, c=colors[i], label=labels[i])

centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], s=200, c='black', marker='X', label='Centroids')
plt.title('Customer Segments using K-Means (k=5)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Task 4: Hierarchical Clustering and DBSCAN

In [ ]:
# 1. Hierarchical Clustering Dendrogram
linkage_matrix = linkage(X_scaled, method='ward')
plt.figure(figsize=(8, 4))
dendrogram(linkage_matrix, truncate_mode='lastp', p=25)
plt.title('Dendrogram for Hierarchical Clustering')
plt.xlabel('Clusters')
plt.ylabel('Euclidean Distance')
plt.tight_layout()
plt.show()

# Agglomerative clustering
hierarchical = AgglomerativeClustering(n_clusters=5, linkage='ward')
hc_labels = hierarchical.fit_predict(X_scaled)
hc_score = silhouette_score(X_scaled, hc_labels)
print(f'Hierarchical Clustering Silhouette Score: {hc_score:.4f}')

# Plot hierarchical clusters
plt.figure(figsize=(7, 5))
for i in range(5):
    plt.scatter(X[hc_labels == i, 0], X[hc_labels == i, 1], s=50, c=colors[i], label=f'Cluster {i}')
plt.title('Hierarchical Clustering (k=5)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2. DBSCAN Clustering
dbscan = DBSCAN(eps=0.35, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db = (db_labels == -1).sum()
print(f'DBSCAN clusters found: {n_clusters_db}, Noise/outliers: {n_noise_db}')

# Plot DBSCAN clusters
plt.figure(figsize=(7, 5))
for k in set(db_labels):
    if k == -1:
        plt.scatter(X[db_labels == k, 0], X[db_labels == k, 1], c='black', marker='x', s=60, label='Outliers / Noise')
    else:
        plt.scatter(X[db_labels == k, 0], X[db_labels == k, 1], s=50, label=f'Cluster {k}')
plt.title('DBSCAN Clustering (eps=0.35, min_samples=5)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Side-by-side Model Comparison
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
for i in range(5):
    plt.scatter(X[kmeans_labels == i, 0], X[kmeans_labels == i, 1], s=30)
plt.title(f'K-Means (k=5)\nScore: {km_score:.3f}')
plt.xlabel('Income')
plt.ylabel('Score')

plt.subplot(1, 3, 2)
for i in range(5):
    plt.scatter(X[hc_labels == i, 0], X[hc_labels == i, 1], s=30)
plt.title(f'Hierarchical (k=5)\nScore: {hc_score:.3f}')
plt.xlabel('Income')

plt.subplot(1, 3, 3)
for k in set(db_labels):
    if k == -1:
        plt.scatter(X[db_labels == k, 0], X[db_labels == k, 1], c='black', marker='x', s=30)
    else:
        plt.scatter(X[db_labels == k, 0], X[db_labels == k, 1], s=30)
plt.title(f'DBSCAN\nClusters: {n_clusters_db}, Noise: {n_noise_db}')
plt.xlabel('Income')

plt.tight_layout()
plt.show()

## Task 5: Groceries Market Basket Analysis from GitHub (Apriori vs FP-Growth)

In [ ]:
# Load Groceries dataset directly through GitHub URL
groceries_url = 'https://raw.githubusercontent.com/ayankhan96686-create/ml-hand-on-lab-4/main/Groceries_dataset.csv'
try:
    groceries = pd.read_csv(groceries_url)
    print('Loaded Groceries dataset from GitHub successfully!')
except Exception:
    groceries = pd.read_csv('Groceries_dataset.csv')
    print('Loaded from local file fallback.')

print('\nGroceries sample:')
display(groceries.head())

# Group items by member and date into transactions
baskets = groceries.groupby(['Member_number', 'Date'])['itemDescription'].apply(list).tolist()
print('Total transactions:', len(baskets))

# Encode baskets into binary format
te = TransactionEncoder()
te_data = te.fit(baskets).transform(baskets)
df_trans = pd.DataFrame(te_data, columns=te.columns_)

# 1. Apriori
t0 = time.time()
freq_items_apriori = apriori(df_trans, min_support=0.005, use_colnames=True)
apriori_time = time.time() - t0
rules_apriori = association_rules(freq_items_apriori, metric='confidence', min_threshold=0.10)

# 2. FP-Growth
t0 = time.time()
freq_items_fpgrowth = fpgrowth(df_trans, min_support=0.005, use_colnames=True)
fpgrowth_time = time.time() - t0
rules_fpgrowth = association_rules(freq_items_fpgrowth, metric='confidence', min_threshold=0.10)

print(f'Apriori Time:   {apriori_time:.4f} s | Itemsets: {len(freq_items_apriori)} | Rules: {len(rules_apriori)}')
print(f'FP-Growth Time: {fpgrowth_time:.4f} s | Itemsets: {len(freq_items_fpgrowth)} | Rules: {len(rules_fpgrowth)}')
print(f'FP-Growth speedup: {apriori_time/fpgrowth_time:.2f}x faster!')

# Top rules sorted by Lift
top_rules = rules_apriori.sort_values(by='lift', ascending=False).head(10).copy()
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))
display(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

## Conclusion
In this lab, we successfully applied clustering and association rule mining on retail datasets. Using K-Means and Hierarchical clustering on the Mall Customers dataset, we determined that k=5 clusters cleanly segments customers into five distinct personas based on their income and spending score (Silhouette Score ~0.555). DBSCAN identified core dense segments and isolated 23 outliers as noise.

For the Groceries dataset, both Apriori and FP-Growth mined identical frequent itemsets and association rules, but FP-Growth was more than 2x faster due to its compressed tree structure avoiding candidate generation. High-lift rules identified valuable cross-selling relationships (such as meats with vegetables and dairy), providing clear insights for retail layout and promotions.